# ⚽ FIFA World Cup Intelligence & Prediction System
## Phase 4: Country Performance Analytics

**Notebook:** `04_country_performance_analytics.ipynb`  
**Input:** `data/matches_prepared.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../data/matches_prepared.csv')

print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

In [ ]:
# Quick column review
print('Columns:')
print(df.columns.tolist())
print('\nSample record:')
df.head(3)

---
## 1: Country Performance Dataset Creation

Each match involves two countries. To build a country-level table, we process each match **twice** — once from the home team's perspective and once from the away team's perspective — then aggregate.

In [ ]:
# Build home perspective records 
home_records = pd.DataFrame({
    'country':        df['home_team'],
    'goals_scored':   df['home_score'],
    'goals_conceded': df['away_score'],
    'win':            (df['match_result'] == 'Home Team Win').astype(int),
    'draw':           (df['match_result'] == 'Draw').astype(int),
    'loss':           (df['match_result'] == 'Away Team Win').astype(int),
})

# Build away perspective records 
away_records = pd.DataFrame({
    'country':        df['away_team'],
    'goals_scored':   df['away_score'],
    'goals_conceded': df['home_score'],
    'win':            (df['match_result'] == 'Away Team Win').astype(int),
    'draw':           (df['match_result'] == 'Draw').astype(int),
    'loss':           (df['match_result'] == 'Home Team Win').astype(int),
})

# --- Stack both perspectives and aggregate per country ---
all_records = pd.concat([home_records, away_records], ignore_index=True)

In [ ]:
print(all_records[['win', 'draw', 'loss']].head())
print(all_records[['win', 'draw', 'loss']].sum())

In [ ]:
country_stats = all_records.groupby('country').agg(
    matches_played  = ('win','count'),
    wins            = ('win','sum'),
    draws           = ('draw','sum'),
    losses          = ('loss','sum'),
    goals_scored    = ('goals_scored','sum'),
    goals_conceded  = ('goals_conceded','sum'),
).reset_index()

In [72]:
# Derived columns
country_stats['goal_difference']  = country_stats['goals_scored'] - country_stats['goals_conceded']
country_stats['win_pct']          = (country_stats['wins'] / country_stats['matches_played'] * 100).round(1)
country_stats['goals_per_match']  = (country_stats['goals_scored'] / country_stats['matches_played']).round(2)
country_stats['conceded_per_match'] = (country_stats['goals_conceded'] / country_stats['matches_played']).round(2)

print(f'Country-level table: {country_stats.shape[0]} countries')
country_stats.head(10)

Country-level table: 336 countries


,country,matches_played,wins,draws,losses,goals_scored,goals_conceded,goal_difference,win_pct,goals_per_match,conceded_per_match
0,Abkhazia,33,15,13,5,54.0,26.0,28.0,45.5,1.64,0.79
1,Afghanistan,147,36,34,77,142.0,293.0,-151.0,24.5,0.97,1.99
2,Albania,398,110,84,204,377.0,589.0,-212.0,27.6,0.95,1.48
3,Alderney,135,5,2,128,73.0,620.0,-547.0,3.7,0.54,4.59
4,Algeria,618,288,162,168,953.0,616.0,337.0,46.6,1.54,1.00
5,Ambazonia,6,0,1,5,3.0,17.0,-14.0,0.0,0.50,2.83
6,American Samoa,55,4,2,49,33.0,357.0,-324.0,7.3,0.60,6.49
7,Andalusia,13,8,4,1,25.0,14.0,11.0,61.5,1.92,1.08
8,Andorra,229,14,31,184,75.0,543.0,-468.0,6.1,0.33,2.37
9,Angola,415,141,148,126,470.0,428.0,42.0,34.0,1.13,1.03


In [ ]:
# Apply a minimum matches filter for meaningful analysis
# Countries with fewer than 30 matches have too little data for reliable percentages
MIN_MATCHES = 30

country_qualified = country_stats[country_stats['matches_played'] >= MIN_MATCHES].copy()

print(f'Countries with {MIN_MATCHES}+ matches: {len(country_qualified)}')
print(f'Countries filtered out: {len(country_stats) - len(country_qualified)}')

---
## 2: Overall Country Performance

Who has played the most, won the most, and maintained the highest win rates across all of international football history?

In [ ]:
# Top 10 by matches played
top_by_matches = country_qualified.nlargest(10, 'matches_played')[['country', 'matches_played', 'wins', 'draws', 'losses', 'win_pct']]
print('Top 10 Most Matches Played:')
top_by_matches.reset_index(drop=True)

fig = px.bar(
    top_by_matches,
    x='country',
    y='matches_played',
    text = 'matches_played',
    color='wins',
    hover_data=['country', 'matches_played', 'wins', 'draws', 'losses', 'win_pct'],
    title='Top 10 by matches played'
)
fig.show()

In [53]:
# Top 10 by total wins
top_by_wins = country_qualified.nlargest(10, 'wins')[['country', 'wins', 'matches_played', 'win_pct']]
print('Top 10 Most Wins:')
top_by_wins.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_by_wins,
    x='country',
    y='wins',
    text = 'wins',
    color='win_pct',
    hover_data=['country', 'wins', 'matches_played', 'win_pct'],
    title='Top 10 by total wins'
)
fig.show()

Top 10 Most Wins:


In [50]:
# Top 10 by win percentage
top_by_winpct = country_qualified.nlargest(10, 'win_pct')[['country', 'win_pct', 'wins', 'matches_played']]
print('Top 10 Highest Win Percentage:')
top_by_winpct.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_by_winpct,
    x='country',
    y='wins',
    text = 'wins',
    color='win_pct',
    hover_data=['country', 'win_pct', 'wins', 'matches_played'],
    title='Top 10 by win percentage '
)
fig.show()

Top 10 Highest Win Percentage:


In [57]:
# Top 10 by draws
top_by_draws = country_qualified.nlargest(10, 'draws')[['country', 'win_pct', 'draws', 'matches_played']]
print('Top 10 Most Draws:')
top_by_draws.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_by_draws,
    x='country',
    y='draws',
    text = 'draws',
    color='win_pct',
    hover_data=['country', 'win_pct', 'draws', 'matches_played'],
    title='Top 10 by draws'
)
fig.show()


Top 10 Most Draws:


In [82]:
# Top 10 by losses (most losses = lots of matches, often weaker nations)
top_by_losses = country_qualified.nlargest(10, 'losses')[['country', 'losses', 'matches_played', 'win_pct']]
print('Top 10 Most Losses:')
top_by_losses.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_by_losses,
    x='country',
    y='losses',
    text = 'losses',
    color='win_pct',
    hover_data=['country', 'losses', 'matches_played', 'win_pct'],
    title='Top 10 by losses'
)
fig.show()

Top 10 Most Losses:



### Key Findings — Overall Country Performance

- **Brazil, England, Argentina, and Sweden** consistently appear at the top by both total wins and matches played — these are historically the most active nations in international football.
- **Win percentage** rankings reveal a different picture — smaller nations who play selectively can maintain very high win rates. Nations like Brazil, Germany, Spain maintain a win rate of roughly 60%.
- Countries with the most **losses** are generally those who have played many matches but compete against stronger opposition in their confederation.

### Football Insights

Sheer win count favors nations with the longest international football histories. **Brazil's combination of high win count and high win percentage** is what makes their record truly exceptional winning 5 World Cup titles.

---
## 3: Attacking Performance Analysis

Which nations have been the most prolific scorers across international football history?

In [66]:
# Top 10 by total goals scored
top_scorers = country_qualified.nlargest(10, 'goals_scored')[['country', 'goals_scored', 'matches_played', 'goals_per_match']]
print('Top 10 — Most Goals Scored:')
top_scorers.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_scorers,
    x='country',
    y='goals_scored',
    text = 'goals_scored',
    color='matches_played',
    hover_data=['country', 'goals_scored', 'matches_played', 'goals_per_match'],
    title='Top 10 by total goals scored'
)
fig.show()

Top 10 — Most Goals Scored:


In [61]:
# Top 10 by goals per match (most efficient attackers)
top_gpm = country_qualified.nlargest(10, 'goals_per_match')[['country', 'goals_per_match', 'goals_scored', 'matches_played']]
print('Top 10 — Highest Goals Per Match:')
top_gpm.reset_index(drop=True)

# Plot 
fig = px.bar(
    top_gpm,
    x='country',
    y='goals_per_match',
    text = 'goals_per_match',
    color='matches_played',
    hover_data=['country', 'goals_per_match', 'goals_scored', 'matches_played'],
    title='Top 10 by goals per match'
)
fig.show()

Top 10 — Highest Goals Per Match:


### Key Findings — Attacking Performance

- **Total goals scored** is heavily correlated with matches played — long-active nations naturally accumulate more goals over time.
- **Goals per match** reveals which nations are consistently attacking regardless of total volume. 
- Nations with high goals-per-match rates tend to play less than 50 matches and regularly face weaker opposition in qualification and friendlies.

### Football Insights

Goals-per-match rankings should be read alongside opponent quality context. A nation averaging 3.0 goals per match in weak qualifying groups is very different from one maintaining that rate in major tournaments.

---
## 4: Defensive Performance Analysis

Which nations have been the hardest to score against across international football history?

In [81]:
# Top 10 best defensive records — fewest goals conceded 
best_defense = country_qualified.nsmallest(10, 'goals_conceded')[['country', 'goals_conceded', 'matches_played', 'conceded_per_match']]
print('Top 10 — Fewest Goals Conceded (Total):')
best_defense.reset_index(drop=True)

# Plot 
fig = px.bar(
    best_defense,
    x='country',
    y='goals_conceded',
    text = 'goals_conceded',
    color='matches_played',
    hover_data=['country', 'goals_conceded', 'matches_played', 'conceded_per_match'],
    title='Top 10 best defensive records'
)
fig.show()

Top 10 — Fewest Goals Conceded (Total):


In [80]:
# Top 10 by fewest goals conceded per match 
best_defense_rate = country_qualified.nsmallest(10, 'conceded_per_match')[['country', 'conceded_per_match', 'goals_conceded', 'matches_played']]
print('Top 10 — Fewest Goals Conceded Per Match:')
best_defense_rate.reset_index(drop=True)

# Plot 
fig = px.bar(
    best_defense_rate,
    x='conceded_per_match',
    y='country',
    text = 'conceded_per_match',
    color='matches_played',
    hover_data=['country','conceded_per_match', 'goals_conceded', 'matches_played'],
    title='Top 10 by fewest goals conceded per match'
)
fig.show()

Top 10 — Fewest Goals Conceded Per Match:


### Key Findings — Defensive Performance

- **Per-match concession rate** is a far better defensive metric than total goals conceded — nations who have played more matches will naturally concede more in total.
- Countries like Brazil and South Korea played over 1000 matches while maintaining Per-match concession rate of just 0.91
- Nations where `goals_per_match > conceded_per_match` by a large margin are historically dominant — they consistently outscore opponents by a meaningful gap.

### Football Insights

The best teams in football history are not just attacking — they control matches in a way that suppresses opponents. The most elite nations show both high scoring rates and low concession rates simultaneously.

---
## 5: Goal Difference Analysis

Goal difference captures the overall dominance of a nation across all matches — it encodes both attacking output and defensive resilience.

In [73]:
# Add goal difference per match
country_qualified['gd_per_match'] = (country_qualified['goal_difference'] / country_qualified['matches_played']).round(2)
country_qualified.head()

,country,matches_played,wins,draws,losses,goals_scored,goals_conceded,goal_difference,win_pct,goals_per_match,conceded_per_match,gd_per_match
0,Abkhazia,33,15,13,5,54.0,26.0,28.0,45.5,1.64,0.79,0.85
1,Afghanistan,147,36,34,77,142.0,293.0,-151.0,24.5,0.97,1.99,-1.03
2,Albania,398,110,84,204,377.0,589.0,-212.0,27.6,0.95,1.48,-0.53
3,Alderney,135,5,2,128,73.0,620.0,-547.0,3.7,0.54,4.59,-4.05
4,Algeria,618,288,162,168,953.0,616.0,337.0,46.6,1.54,1.00,0.55


In [74]:
# Top 10 positive goal difference (most dominant)
top_gd_pos = country_qualified.nlargest(10, 'goal_difference')[['country', 'goal_difference', 'gd_per_match', 'matches_played']]
print('Top 10 — Best Total Goal Difference:')
top_gd_pos.reset_index(drop=True)

# Top 10 negative goal difference (historically struggled)
top_gd_neg = country_qualified.nsmallest(10, 'goal_difference')[['country', 'goal_difference', 'gd_per_match', 'matches_played']]
print('Top 10 — Worst Total Goal Difference:')
top_gd_neg.reset_index(drop=True)

Top 10 — Best Total Goal Difference:


,country,goal_difference,gd_per_match,matches_played
0,Brazil,1348.0,1.27,1060
1,England,1340.0,1.23,1090
2,Germany,1128.0,1.09,1032
3,Argentina,956.0,0.89,1070
4,Spain,899.0,1.15,784
5,South Korea,879.0,0.87,1008
6,Netherlands,765.0,0.87,880
7,Sweden,754.0,0.68,1102
8,Mexico,715.0,0.71,1004
9,Italy,689.0,0.77,893


In [87]:
top_gd_pos['type'] = 'Best GD'
top_gd_neg['type'] = 'Worst GD'

gd_compare = pd.concat([top_gd_pos, top_gd_neg], ignore_index=True)

fig = px.bar(
    gd_compare,
    x='goal_difference',
    y='country',
    color='type',
    orientation='h',
    text='goal_difference',
    hover_data=['gd_per_match', 'matches_played'],
    title='Top 10 Best and Worst Goal Difference Countries'
)
fig.update_layout(
    height=700,
    bargap=0.15
)
fig.show()

### Key Findings — Goal Difference

- **Total goal difference** again favors high-volume nations. **GD per match is the more comparable metric** across nations with different levels of activity.
- Nations with the best GD per match consistently score 1.0–1.5 more goals than they concede per game — a mark of consistent dominance.
- Nations with large **negative** goal differences have played many matches against stronger competition — not necessarily weak, but outmatched by historical opponents.

### Football Insights

Goal difference per match is one of the cleanest single-number summaries of a nation's historical quality. The nations at the top of this metric are those who have dominated throughout history — not just in one era.

---
## 6: Country Ranking System

We build a simple points-based historical ranking score that captures wins, draws, and goal difference together — similar to how a league table works.

**Formula:**
> `Ranking Score = (Wins × 3) + (Draws × 1) + (Goal Difference × 0.1)`

The `0.1` weight on goal difference prevents it from dominating the score while still rewarding dominant performances. Win points use the standard football system. The score is calculated on raw totals — not per-match — to reward longevity and consistency.

In [107]:
GD_WEIGHT = 0.1

country_qualified = country_qualified.copy()
country_qualified['ranking_score'] = (
    (country_qualified['wins']  * 3) +
    (country_qualified['draws'] * 1) +
    (country_qualified['goal_difference'] * GD_WEIGHT)
).round(1)

country_qualified = country_qualified.sort_values('ranking_score', ascending=False).reset_index(drop=True)

# Top 20 Leaderboard visualization
top20_ranked = country_qualified.nlargest(20,'ranking_score')[[
    'country', 'ranking_score', 'wins', 'draws', 'losses',
    'matches_played', 'win_pct', 'goal_difference'
]].reset_index(drop=True)

# Plot 
fig = px.bar(
    top20_ranked,
    x='ranking_score',
    y='country',
    text = 'ranking_score',
    color='ranking_score',
    hover_data=['country', 'ranking_score', 'wins',
    'matches_played', 'win_pct', 'goal_difference'],
    title='Top 20 by ranking score'
)
fig.update_layout(
    height=700,
    bargap=0.15
)
fig.show()


### Key Findings — Ranking System

- Nations with long histories and consistent win rates score highest.
- The GD component at 0.1 weight adds meaningful differentiation between nations with similar win/draw counts without allowing extremely high-scoring mismatches to inflate scores.
- **Brazil, England, Argentina, and Germany** consistently rank near the top across different weighting schemes — a sign dominance.

### Football Insights

Any historical ranking system has an inherent bias toward older, more active nations. The key takeaway is not the exact rank but the **relative clustering** — there is a clear tier of elite nations, a large middle group, and a long tail of smaller footballing nations.

---
## 7: Home Advantage Analysis

Does playing at home actually increase a team's chance of winning? We use the full match-level dataset to measure this directly.